# ViMamba-SER | Phase B: PhoWhisper transcript → PhoBERT → B1/B2 
> Cấu trúc features mới từ dataset: ['speaker_id', 'path', 'duration', 'accent', 'emotion', 'emotion_id', 'gender']

In [ ]:
# ============================================================
# SETUP: Mount Drive & Cài \u0111ặt dependencies
# ============================================================
!pip install datasets transformers scikit-learn -q

import torch, numpy as np, os, json
from datasets import load_dataset, Audio
from transformers import (
    WavLMModel, Wav2Vec2FeatureExtractor,
    AutoTokenizer, AutoModel,
    pipeline
)
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/ViMamba_SER/midterm_results"
os.makedirs(SAVE_DIR, exist_ok=True)
print("\u2705 Setup and Mount Drive done.")

In [ ]:
# ============================================================
# BƯỚC 1: Load ViSEC
# ============================================================
print("="*55)
print("BƯỚC 1: Load ViSEC...")
print("="*55)
ds = load_dataset("hustep-lab/ViSEC", split="train")

# Cast column 'path' sang Audio để code dễ xử lý numpy array sau này
ds = ds.cast_column("path", Audio(sampling_rate=16000))

print(f"\u2705 {len(ds)} samples, labels: {set(ds['emotion'])}")
print(f"   Xác nhận array shape ds[0]: {ds[0]['path']['array'].shape}")

In [ ]:
# ============================================================
# BƯỚC 2: PhoWhisper → sinh transcript toàn bộ 5280 samples
# ============================================================
print("\n" + "="*55)
print("BƯỚC 2: Sinh transcript bằng PhoWhisper-base...")
print("Ước tính: 2–3 giờ trên T4. Để máy chạy.")
print("="*55)

TRANSCRIPT_CACHE = f"{SAVE_DIR}/transcripts.json"

if os.path.exists(TRANSCRIPT_CACHE):
    print("\u2705 Tìm thấy cache transcript, load lại...")
    with open(TRANSCRIPT_CACHE) as f:
        transcripts = json.load(f)
else:
    asr = pipeline(
        "automatic-speech-recognition",
        model="vinai/PhoWhisper-base",
        device=0,
        chunk_length_s=30,
        batch_size=16   # T4 có thể handle 16
    )

    transcripts = []
    checkpoint_every = 200

    for i, item in enumerate(ds):
        result = asr(item["path"]["array"])
        transcripts.append(result["text"])

        # In tiến độ mỗi 100 samples
        if (i+1) % 100 == 0:
            print(f"  [{i+1}/{len(ds)}] {result['text'][:60]}...")

        # Lưu checkpoint mỗi 200 samples — tránh mất nếu crash
        if (i+1) % checkpoint_every == 0:
            with open(TRANSCRIPT_CACHE, "w", encoding="utf-8") as f:
                json.dump(transcripts, f, ensure_ascii=False)
            print(f"  \U0001f4be Checkpoint lưu tại {i+1} samples")

    # Lưu lần cuối
    with open(TRANSCRIPT_CACHE, "w", encoding="utf-8") as f:
        json.dump(transcripts, f, ensure_ascii=False)
    print(f"\u2705 Xong! Đã lưu {len(transcripts)} transcripts → {TRANSCRIPT_CACHE}")

# Kiểm tra vài transcript
print("\nVí dụ transcript:")
for i in [0, 100, 500]:
    if i < len(transcripts):
        print(f"  [{i}] label={ds[i]['emotion']} | text={transcripts[i][:80]}")

In [ ]:
# ============================================================
# BƯỚC 3: WavLM embeddings (load từ cache nếu đã có)
# ============================================================
print("\n" + "="*55)
print("BƯỚC 3: Trích WavLM embeddings...")
print("="*55)

WAVLM_CACHE = f"{SAVE_DIR}/wavlm_embeddings.npy"
LABEL_CACHE  = f"{SAVE_DIR}/labels.npy"

if os.path.exists(WAVLM_CACHE):
    print("\u2705 Load WavLM cache...")
    X_audio = np.load(WAVLM_CACHE)
    labels_raw = np.load(LABEL_CACHE, allow_pickle=True)
else:
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base")
    wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base").cuda().eval()

    def batch_embed_audio(batch):
        inputs = feature_extractor(
            [x["array"] for x in batch["path"]],
            sampling_rate=16000, return_tensors="pt", padding=True
        )
        with torch.no_grad():
            out = wavlm(inputs.input_values.cuda())
            emb = out.last_hidden_state.mean(dim=1).cpu().numpy()
        return {"embedding": list(emb)}

    ds_audio = ds.map(batch_embed_audio, batched=True, batch_size=32,
                      desc="WavLM embeddings")
    X_audio = np.array(ds_audio["embedding"])
    labels_raw = np.array(ds["emotion"])
    np.save(WAVLM_CACHE, X_audio)
    np.save(LABEL_CACHE, labels_raw)
    print(f"\u2705 WavLM embeddings: {X_audio.shape}")

le = LabelEncoder()
y = le.fit_transform(labels_raw)
print(f"\u2705 Labels: {le.classes_}")

In [ ]:
# ============================================================
# BƯỚC 4: PhoBERT embeddings từ transcript thật
# ============================================================
print("\n" + "="*55)
print("BƯỚC 4: Trích PhoBERT embeddings từ transcript thật...")
print("="*55)

PHOBERT_CACHE = f"{SAVE_DIR}/phobert_embeddings.npy"

if os.path.exists(PHOBERT_CACHE):
    print("\u2705 Load PhoBERT cache...")
    X_text = np.load(PHOBERT_CACHE)
else:
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
    phobert   = AutoModel.from_pretrained("vinai/phobert-base-v2").cuda().eval()

    X_text = []
    batch_size = 32
    for i in range(0, len(transcripts), batch_size):
        batch = transcripts[i:i+batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=128)
        enc = {k: v.cuda() for k, v in enc.items()}
        with torch.no_grad():
            out = phobert(**enc)
            emb = out.last_hidden_state[:, 0, :].cpu().numpy()  # [CLS] token
        X_text.extend(emb)
        if (i // batch_size + 1) % 20 == 0:
            print(f"  [{min(i+batch_size, len(transcripts))}/{len(transcripts)}]")

    X_text = np.array(X_text)
    np.save(PHOBERT_CACHE, X_text)
    print(f"\u2705 PhoBERT embeddings: {X_text.shape}")

X_fusion = np.concatenate([X_audio, X_text], axis=1)
print(f"\u2705 Fusion shape: {X_fusion.shape}  (768 audio + 768 text = 1536)")

In [ ]:
# ============================================================
# BƯỚC 5: Train B1 và B2
# ============================================================
def make_mlp(input_dim, num_classes=4, dropout=0.3):
    return nn.Sequential(
        nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(256, 64),        nn.ReLU(),
        nn.Linear(64, num_classes)
    )

def run_and_report(X, y, name, input_dim, seed=42, epochs=50):
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=0.30, random_state=seed, stratify=y)
    X_val, X_te, y_val, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=seed, stratify=y_tmp)

    def loader(a, b, shuffle=False):
        return DataLoader(TensorDataset(torch.FloatTensor(a),
                                        torch.LongTensor(b)),
                          batch_size=64, shuffle=shuffle)

    model = make_mlp(input_dim).cuda()
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    crit  = nn.CrossEntropyLoss()

    best_val, best_test, best_preds = 0, 0, None

    for epoch in range(epochs):
        model.train()
        for xb, yb in loader(X_tr, y_tr, shuffle=True):
            opt.zero_grad()
            crit(model(xb.cuda()), yb.cuda()).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_acc = sum((model(xb.cuda()).argmax(1)==yb.cuda()).sum().item()
                          for xb,yb in loader(X_val, y_val)) / len(y_val)
            if val_acc > best_val:
                best_val = val_acc
                all_preds, all_true = [], []
                for xb, yb in loader(X_te, y_te):
                    all_preds.extend(model(xb.cuda()).argmax(1).cpu().numpy())
                    all_true.extend(yb.numpy())
                best_test  = sum(p==t for p,t in zip(all_preds,all_true)) / len(y_te)
                best_preds = (all_preds, all_true)

        if (epoch+1) % 10 == 0:
            print(f"  Epoch {epoch+1:2d}/{epochs} | val={val_acc:.4f} | best_test={best_test:.4f}")

    print(f"\n>>> [{name}] Test Accuracy: {best_test*100:.2f}%")
    print(classification_report(best_preds[1], best_preds[0],
                                 target_names=le.classes_))
    return best_test

print("\n" + "="*55)
print("B1: PhoBERT transcript thật → MLP (text-only)")
print("="*55)
b1_acc = run_and_report(X_text, y, "B1 Text-only", input_dim=768)

print("\n" + "="*55)
print("B2: WavLM + PhoBERT transcript thật → MLP (fusion)")
print("="*55)
b2_acc = run_and_report(X_fusion, y, "B2 Audio+Text fusion", input_dim=1536)

In [ ]:
# ============================================================
# BƯỚC 6: Bảng tổng kết + lưu Drive
# ============================================================
# Load A1 từ kết quả trước (giả sử \u0111ã có kết quả Phase A audio-only baseline)
a1_acc = 0.5644  # kết quả A1 đã có để so sánh

summary = {
    "A1_audio_only_MLP":         round(a1_acc * 100, 2),
    "B1_text_only_transcript":   round(b1_acc * 100, 2),
    "B2_fusion_transcript":      round(b2_acc * 100, 2),
    "delta_B2_vs_A1":            round((b2_acc - a1_acc) * 100, 2),
}

print("\n" + "="*55)
print("TỔNG KẾT SO SÁNH")
print("="*55)
print(f"{'A1 Audio-only MLP':<30} {a1_acc*100:>8.2f}%")
print(f"{'B1 Text-only (transcript)':<30} {b1_acc*100:>8.2f}%")
print(f"{'B2 Audio+Text fusion':<30} {b2_acc*100:>8.2f}%")
print(f"{'Delta (B2 - A1)':<30} {(b2_acc-a1_acc)*100:>+8.2f}%")
print("="*55)

if b2_acc > a1_acc:
    print("\u2705 Text enhancement CÓ giúp ích — B2 > A1")
else:
    print("\u26a0\ufe0f  Text chưa giúp ích với simple concat — cần TME module (final)")

with open(f"{SAVE_DIR}/phase_B_results.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"\n\U0001f4be Đã lưu → {SAVE_DIR}/phase_B_results.json")